# Annexe B — Cahier de code, Chapitre 4
## Métriques de segmentation

Ce notebook accompagne le chapitre 4 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Vérité terrain et prédiction (deux masques qui se recouvrent en partie)
import numpy as np
gt   = np.zeros((100, 100), dtype=bool); gt[20:70, 20:70] = True
pred = np.zeros((100, 100), dtype=bool); pred[30:80, 30:80] = True

## 4.1 — IoU

In [ ]:
# intersection / union
inter = (gt & pred).sum()
union = (gt | pred).sum()
print(inter / union)

## 4.2 — Coefficient de Dice

In [ ]:
# 2·intersection / (|A| + |B|)
print(2 * inter / (gt.sum() + pred.sum()))

## 4.3 — Précision, rappel et F1

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
y_true, y_pred = gt.ravel(), pred.ravel()
print(precision_score(y_true, y_pred))
print(recall_score(y_true, y_pred))
print(f1_score(y_true, y_pred))

## 4.4 — Average Precision (AP)

In [ ]:
# moyenne de la précision sur tous les seuils, à partir de scores continus
from sklearn.metrics import average_precision_score
scores = np.random.rand(gt.size)          # scores du modèle (factice)
print(average_precision_score(gt.ravel(), scores))

## 4.5 — Panoptic Quality (PQ)

In [ ]:
# PQ = (Σ IoU des vrais positifs) / (TP + ½FP + ½FN), version 1 objet
iou = inter / union
TP = 1 if iou > 0.5 else 0
FP, FN = 1 - TP, 1 - TP
print((iou * TP) / (TP + 0.5 * FP + 0.5 * FN))

## 4.6 — Boundary F1 (BF)

In [ ]:
# F1 calculé sur les bords (avec tolérance via dilatation)
from skimage.segmentation import find_boundaries
from scipy.ndimage import binary_dilation
bg, bp = find_boundaries(gt), find_boundaries(pred)
tol = 2
match_p = (bp & binary_dilation(bg, iterations=tol)).sum() / bp.sum()
match_r = (bg & binary_dilation(bp, iterations=tol)).sum() / bg.sum()
print(2 * match_p * match_r / (match_p + match_r))